# Notebook 3 — Gold Layer KPIs

**Project:** Tractor Industry Analytics  
**Layer:** Gold (analytical aggregates → dashboard-ready Delta tables)  
**Author:** Tractor Analytics Pipeline  
**Last Updated:** 2026-06-18

## Overview
This notebook reads Silver tables and produces KPI-level Gold tables consumed by Looker Studio
and the Next.js web report.

### Gold Tables Produced
| Gold Table | Source Silver Table(s) | Description |
|---|---|---|
| `gold.market_summary` | `global_tractor_sales` | Global volume by year, top-5 countries, CAGR |
| `gold.india_seasonality` | `india_tractor_sales` + `india_tractor_seasonality` | Monthly demand estimates with season flags |
| `gold.hp_segment_trend` | `us_aem_tractor_sales_hp` | US HP category mix over time |
| `gold.price_vs_sales` | `fred_annual_indices` + `global_tractor_sales` | Equipment PPI / farm income vs. units sold |
| `gold.top_exporters` | `comtrade_hs8701_exports` | Top-10 countries by FOB value 2015–2022 |
| `gold.india_brand_share` | `india_tractor_brands` | Brand share ranking for dashboards |

## Prerequisites
- **Notebook 02** (`02_silver_transformation.ipynb`) must have been run successfully.
- All `silver.*` tables must exist in the metastore.
- Cluster: Databricks Runtime 13.x+ (Delta Lake 2.x, PySpark 3.4+).

## Cell 1 — Environment Setup

In [ ]:
import os
from datetime import datetime, timezone

# ── Runtime detection ───────────────────────────────────────────────────────
try:
    dbutils  # noqa: F821
    ON_DATABRICKS = True
except NameError:
    ON_DATABRICKS = False

print(f"Running on Databricks: {ON_DATABRICKS}")

SILVER_SCHEMA  = "silver"
GOLD_SCHEMA    = "gold"
AGGREGATED_AT  = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Aggregation timestamp : {AGGREGATED_AT}")

## Cell 2 — Spark Session & Gold Schema Bootstrap

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType
)

if ON_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()
else:
    spark = (
        SparkSession.builder
        .appName("gold_kpis_local")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

spark.conf.set("spark.sql.shuffle.partitions", "8")

# Create gold schema (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"USE {GOLD_SCHEMA}")
print(f"Active schema: {GOLD_SCHEMA}")

## Cell 3 — Helper: Write Gold Table

Appends standard gold-layer lineage columns and writes as Delta (overwrite = idempotent).

In [ ]:
def write_gold(df, table_name: str, description: str = "") -> None:
    """
    Append gold lineage columns and write as a Delta table.

    Parameters
    ----------
    df         : aggregated Spark DataFrame
    table_name : target table name (without schema prefix)
    description: short description for log output
    """
    full_table = f"{GOLD_SCHEMA}.{table_name}"

    df = (
        df
        .withColumn("_aggregated_at", F.lit(AGGREGATED_AT))
        .withColumn("_gold_layer",    F.lit(True))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    count = spark.table(full_table).count()
    label = description or table_name
    print(f"[OK] {full_table:<50s}  {count:>7,} rows   {label}")

## Cell 4 — `gold.market_summary`

**Source:** `silver.global_tractor_sales`  
**Grain:** One row per (year, country) for the `hp_category = 'all'` rows, plus a global total row.

**Columns produced:**

| Column | Description |
|---|---|
| `year` | Calendar year |
| `country` | Country name or `'World'` for the global total row |
| `units_sold` | Total retail / domestic units |
| `global_units` | World total for that year (useful for share calculation in BI) |
| `country_share_pct` | Country's share of global total (%) |
| `yoy_growth_pct` | Year-over-year growth per country |
| `is_top5` | Boolean — country is in the top-5 by cumulative units over all years |
| `cagr_5yr_pct` | 5-year CAGR ending in this year for each country |

**Approach:**
1. Filter `silver.global_tractor_sales` to `hp_category = 'all'` to avoid double-counting the US.
2. Compute a global total row per year by summing all countries.
3. Join country rows with the global total to compute share %.
4. Compute YoY growth and 5-year CAGR per country.

In [ ]:
# ── Base: country-year totals (all-HP rows only to avoid double-counting) ────
country_year = (
    spark.table(f"{SILVER_SCHEMA}.global_tractor_sales")
    .filter(F.col("hp_category") == "all")
    .filter(F.col("units_sold").isNotNull() & (F.col("units_sold") > 0))
    .groupBy("year", "country", "country_code")
    .agg(
        F.sum("units_sold").cast(LongType()).alias("units_sold"),
        F.first("data_type").alias("data_type")
    )
)

# ── Global total per year ────────────────────────────────────────────────────
global_year = (
    country_year
    .groupBy("year")
    .agg(F.sum("units_sold").cast(LongType()).alias("global_units"))
)

# ── Join and compute share ───────────────────────────────────────────────────
w_country = Window.partitionBy("country").orderBy("year")

country_joined = (
    country_year
    .join(global_year, "year", "left")
    .withColumn("country_share_pct",
                F.round(F.col("units_sold").cast(DoubleType())
                        / F.col("global_units") * 100, 2))
    # YoY growth per country
    .withColumn("prev_units",      F.lag("units_sold", 1).over(w_country))
    .withColumn("yoy_growth_pct",
                F.round(
                    (F.col("units_sold") - F.col("prev_units"))
                    / F.col("prev_units") * 100,
                    2
                ))
    # 5-year CAGR: (units_t / units_t-5)^(1/5) - 1
    .withColumn("units_5yr_ago",   F.lag("units_sold", 5).over(w_country))
    .withColumn("cagr_5yr_pct",
                F.when(F.col("units_5yr_ago").isNotNull() & (F.col("units_5yr_ago") > 0),
                       F.round(
                           (F.pow(
                               F.col("units_sold").cast(DoubleType())
                               / F.col("units_5yr_ago").cast(DoubleType()),
                               F.lit(0.2)   # 1/5
                           ) - 1) * 100,
                           2
                       ))
                 .otherwise(F.lit(None).cast(DoubleType())))
    .drop("prev_units", "units_5yr_ago")
)

# ── Identify top-5 countries by cumulative units ─────────────────────────────
top5_countries = (
    country_year
    .groupBy("country")
    .agg(F.sum("units_sold").alias("cumulative_units"))
    .orderBy(F.desc("cumulative_units"))
    .limit(5)
    .select("country")
    .withColumn("is_top5", F.lit(True))
)

country_final = (
    country_joined
    .join(top5_countries, "country", "left")
    .withColumn("is_top5", F.coalesce(F.col("is_top5"), F.lit(False)))
)

# ── Append a 'World' summary row ─────────────────────────────────────────────
w_world = Window.orderBy("year")

world_row = (
    global_year
    .withColumn("country",          F.lit("World"))
    .withColumn("country_code",     F.lit("WLD"))
    .withColumn("data_type",        F.lit("aggregate"))
    .withColumn("units_sold",       F.col("global_units"))
    .withColumn("country_share_pct", F.lit(100.0))
    .withColumn("prev_units",       F.lag("global_units", 1).over(w_world))
    .withColumn("yoy_growth_pct",
                F.round(
                    (F.col("global_units") - F.col("prev_units"))
                    / F.col("prev_units") * 100,
                    2
                ))
    .withColumn("units_5yr_ago",    F.lag("global_units", 5).over(w_world))
    .withColumn("cagr_5yr_pct",
                F.when(F.col("units_5yr_ago").isNotNull() & (F.col("units_5yr_ago") > 0),
                       F.round(
                           (F.pow(
                               F.col("global_units").cast(DoubleType())
                               / F.col("units_5yr_ago").cast(DoubleType()),
                               F.lit(0.2)
                           ) - 1) * 100,
                           2
                       ))
                 .otherwise(F.lit(None).cast(DoubleType())))
    .withColumn("is_top5", F.lit(True))
    .drop("prev_units", "units_5yr_ago")
    .select(
        "year", "country", "country_code", "data_type",
        "units_sold", "global_units", "country_share_pct",
        "yoy_growth_pct", "cagr_5yr_pct", "is_top5"
    )
)

market_summary = (
    country_final
    .select(
        "year", "country", "country_code", "data_type",
        "units_sold", "global_units", "country_share_pct",
        "yoy_growth_pct", "cagr_5yr_pct", "is_top5"
    )
    .union(world_row)
    .orderBy("year", "country")
)

write_gold(market_summary, "market_summary",
           "Global market: country-year volumes, share %, YoY growth, 5yr CAGR")

print("\nTop-5 countries — latest year:")
spark.table("gold.market_summary") \
    .filter(F.col("is_top5") & (F.col("country") != "World")) \
    .orderBy(F.desc("year"), F.desc("units_sold")) \
    .select("year", "country", "units_sold", "country_share_pct",
            "yoy_growth_pct", "cagr_5yr_pct") \
    .show(15, truncate=False)

print("\nWorld total by year:")
spark.table("gold.market_summary") \
    .filter(F.col("country") == "World") \
    .orderBy(F.desc("year")) \
    .select("year", "units_sold", "yoy_growth_pct", "cagr_5yr_pct") \
    .show(10, truncate=False)

## Cell 5 — `gold.india_seasonality`

**Sources:** `silver.india_tractor_sales` + `silver.india_tractor_seasonality`  
**Grain:** One row per (fiscal_year, month).  

**Approach:**
1. Cross-join each annual sales figure with the 12 monthly share percentages.
2. Compute `estimated_monthly_units` = `total_units_sold × avg_share_pct / 100`.
3. Carry over season labels and cumulative share for chart ordering.
4. Add `fiscal_month_label` = 'Apr 2023', 'May 2023', … for time-series axis.

| Column | Description |
|---|---|
| `fiscal_year` | FY string e.g. `2023-24` |
| `calendar_year` | Dominant calendar year |
| `fiscal_month` | Month within FY (1=April … 12=March) |
| `calendar_month` | Calendar month (1=Jan … 12=Dec) |
| `month_name` | e.g. `April` |
| `month_label` | e.g. `Apr 2023` — for display on chart axis |
| `avg_share_pct` | Historical share % for this month |
| `estimated_monthly_units` | Estimated units = annual × share% |
| `season_label` | e.g. `Kharif Sowing` |
| `season_group` | `HIGH` / `MEDIUM` / `LOW` |
| `yoy_growth_pct` | Annual YoY growth inherited from silver |

In [ ]:
annual    = spark.table(f"{SILVER_SCHEMA}.india_tractor_sales")
monthly   = spark.table(f"{SILVER_SCHEMA}.india_tractor_seasonality")

# Cross-join: every annual year × every month (12 rows per year)
# Use broadcast hint for the small (12-row) monthly table
india_seasonality = (
    annual
    .crossJoin(F.broadcast(monthly))
    # estimated monthly units = annual total × monthly share %
    .withColumn("estimated_monthly_units",
                F.round(
                    F.col("total_units_sold").cast(DoubleType())
                    * F.col("avg_share_pct") / 100,
                    0
                ).cast(LongType()))
    # Month label for BI axis: e.g. 'Apr 2023'
    # fiscal_month 1 = April of calendar_year - 1
    # April..December → calendar_year - 1;  January..March → calendar_year
    .withColumn("label_year",
                F.when(F.col("calendar_month") >= 4,
                       F.col("calendar_year") - 1)
                 .otherwise(F.col("calendar_year")))
    .withColumn("month_label",
                F.concat(
                    F.date_format(
                        F.to_date(
                            F.concat_ws("-",
                                        F.col("label_year"),
                                        F.lpad(F.col("calendar_month").cast(StringType()), 2, "0"),
                                        F.lit("01")
                            ),
                            "yyyy-MM-dd"
                        ),
                        "MMM yyyy"
                    )
                ))
    # Drop silver lineage columns not needed in gold
    .drop("_transformed_at", "_silver_layer",
          "cumulative_share_pct", "label_year", "notes", "source")
    .select(
        "fiscal_year", "year_start", "year_end", "calendar_year",
        F.col("month").alias("fiscal_month"),
        "calendar_month", "month_name", "month_label",
        "total_units_sold", "avg_share_pct", "estimated_monthly_units",
        "season_label", "season_group",
        "yoy_growth_pct", "rolling_avg_3yr", "is_provisional"
    )
    .orderBy("year_start", "fiscal_month")
)

write_gold(india_seasonality, "india_seasonality",
           "India monthly estimated units = annual × seasonal share %")

print("\nLatest fiscal year — monthly distribution:")
spark.table("gold.india_seasonality") \
    .orderBy(F.desc("year_start"), "fiscal_month") \
    .filter(F.col("year_start") == spark.table("gold.india_seasonality")
            .agg(F.max("year_start")).collect()[0][0]) \
    .select("fiscal_year", "month_name", "month_label",
            "avg_share_pct", "estimated_monthly_units", "season_group") \
    .show(12, truncate=False)

# Summary: average monthly units by season group across all years
print("\nAverage monthly units by season group (all years):")
spark.table("gold.india_seasonality") \
    .groupBy("season_group") \
    .agg(
        F.round(F.avg("estimated_monthly_units"), 0).alias("avg_monthly_units"),
        F.round(F.avg("avg_share_pct"), 2).alias("avg_share_pct")
    ) \
    .orderBy(F.desc("avg_monthly_units")) \
    .show(truncate=False)

## Cell 6 — `gold.hp_segment_trend`

**Source:** `silver.us_aem_tractor_sales_hp`  
**Grain:** One row per (year, hp_category) — long/tidy format for BI tools.

Pivoting to long format makes it easy to filter or colour by `hp_category` in any BI tool
without writing CASE statements in the dashboard.

| Column | Description |
|---|---|
| `year` | Calendar year |
| `hp_category` | `<40HP` / `40-100HP` / `100+HP` / `4WD` |
| `units_sold` | Retail units for that HP segment |
| `share_pct` | HP segment share of total 2WD+4WD |
| `total_units` | US market total for that year |
| `yoy_segment_growth_pct` | YoY growth specific to each HP segment |
| `dominant_segment` | Segment with highest units that year |
| `is_provisional` | True if data is YTD / provisional |

In [ ]:
us_hp = spark.table(f"{SILVER_SCHEMA}.us_aem_tractor_sales_hp")

# Pivot to long format — one row per year × hp_category
# We stack the three main segments plus 4WD as a separate row

def hp_segment_rows(df, unit_col, share_col, category_label):
    """Extract one HP segment into a standardised long-format DataFrame."""
    w_seg = Window.partitionBy(F.lit(category_label)).orderBy("year")
    return (
        df
        .select(
            "year",
            F.lit(category_label).alias("hp_category"),
            F.col(unit_col).alias("units_sold"),
            F.col(share_col).alias("share_pct"),
            "total_units", "dominant_segment", "is_provisional"
        )
        .withColumn("prev_units", F.lag("units_sold", 1).over(w_seg))
        .withColumn("yoy_segment_growth_pct",
                    F.round(
                        (F.col("units_sold") - F.col("prev_units"))
                        / F.col("prev_units") * 100,
                        2
                    ))
        .drop("prev_units")
    )

seg_under40 = hp_segment_rows(us_hp, "under_40hp_units",  "under_40hp_share_pct",  "<40HP")
seg_mid     = hp_segment_rows(us_hp, "hp40_to_100_units", "hp40_to_100_share_pct", "40-100HP")
seg_over100 = hp_segment_rows(us_hp, "over_100hp_units",  "over_100hp_share_pct",  "100+HP")

# 4WD share is not a direct percentage of total — compute it
seg_4wd = (
    us_hp
    .select(
        "year",
        F.lit("4WD").alias("hp_category"),
        F.col("hp_4wd_units").alias("units_sold"),
        F.round(
            F.col("hp_4wd_units").cast(DoubleType())
            / F.col("total_units") * 100, 1
        ).alias("share_pct"),
        "total_units", "dominant_segment", "is_provisional"
    )
    .withColumn("prev_units", F.lag("units_sold", 1)
                .over(Window.partitionBy(F.lit("4WD")).orderBy("year")))
    .withColumn("yoy_segment_growth_pct",
                F.round(
                    (F.col("units_sold") - F.col("prev_units"))
                    / F.col("prev_units") * 100,
                    2
                ))
    .drop("prev_units")
)

hp_trend = (
    seg_under40
    .union(seg_mid)
    .union(seg_over100)
    .union(seg_4wd)
    .orderBy("year", "hp_category")
)

write_gold(hp_trend, "hp_segment_trend",
           "US HP segment trend — long format, YoY growth per segment")

print("\nHP segment trend (most recent 3 years):")
max_yr = spark.table("gold.hp_segment_trend").agg(F.max("year")).collect()[0][0]
spark.table("gold.hp_segment_trend") \
    .filter(F.col("year") >= max_yr - 2) \
    .orderBy("year", "hp_category") \
    .select("year", "hp_category", "units_sold", "share_pct",
            "total_units", "yoy_segment_growth_pct", "dominant_segment") \
    .show(20, truncate=False)

print("\nDominant segment frequency by year:")
spark.table("gold.hp_segment_trend") \
    .filter(F.col("hp_category") == "<40HP") \
    .groupBy("dominant_segment") \
    .agg(F.count("*").alias("years_as_dominant")) \
    .orderBy(F.desc("years_as_dominant")) \
    .show(truncate=False)

## Cell 7 — `gold.price_vs_sales`

**Sources:** `silver.fred_annual_indices` + `silver.global_tractor_sales`  
**Grain:** One row per year with aligned economic indicators and market volumes.

This table powers the demand-driver correlation chart in the dashboard.

| Column | Description |
|---|---|
| `year` | Calendar year |
| `india_units` | India annual tractor units |
| `us_units` | US annual tractor units (all segments) |
| `avg_farm_equip_ppi` | Farm equipment PPI (WPU114) annual average |
| `avg_corn_ppi` | Corn PPI annual average |
| `avg_wheat_ppi` | Wheat PPI annual average |
| `net_farm_income_bn_usd` | US net farm income (BEA, $bn) |
| `farm_equip_ppi_yoy_pct` | Farm equipment PPI YoY change |
| `farm_income_yoy_pct` | Farm income YoY change |
| `india_units_yoy_pct` | India sales YoY change |
| `us_units_yoy_pct` | US sales YoY change |
| `india_ppi_corr_flag` | High-PPI-growth years (equip PPI YoY > 5%) |
| `high_farm_income_flag` | Farm income above 10-year median |

**Note on correlation:** Pearson correlation coefficients are computed as summary statistics
and printed but are NOT stored as rows — they're added as a separate summary DataFrame shown inline.

In [ ]:
fred = spark.table(f"{SILVER_SCHEMA}.fred_annual_indices")

# ── Pull India and US annual units from the unified global sales table ────────
sales_pivot = (
    spark.table(f"{SILVER_SCHEMA}.global_tractor_sales")
    .filter(F.col("hp_category") == "all")
    .filter(F.col("units_sold").isNotNull() & (F.col("units_sold") > 0))
    .groupBy("year")
    .pivot("country", ["India", "United States"])
    .agg(F.sum("units_sold"))
    .withColumnRenamed("India",         "india_units")
    .withColumnRenamed("United States", "us_units")
)

# ── Join FRED indices with sales ─────────────────────────────────────────────
w_yr = Window.orderBy("year")

price_vs_sales = (
    fred
    .join(sales_pivot, "year", "left")
    # YoY changes for sales
    .withColumn("prev_india",   F.lag("india_units", 1).over(w_yr))
    .withColumn("prev_us",      F.lag("us_units",    1).over(w_yr))
    .withColumn("india_units_yoy_pct",
                F.round(
                    (F.col("india_units") - F.col("prev_india"))
                    / F.col("prev_india") * 100,
                    2
                ))
    .withColumn("us_units_yoy_pct",
                F.round(
                    (F.col("us_units") - F.col("prev_us"))
                    / F.col("prev_us") * 100,
                    2
                ))
    # Rename YoY columns from silver for clarity
    .withColumnRenamed("avg_farm_equip_ppi_yoy_pct",       "farm_equip_ppi_yoy_pct")
    .withColumnRenamed("net_farm_income_bn_usd_yoy_pct",   "farm_income_yoy_pct")
    # Flag: high PPI growth years — equipment became more expensive
    .withColumn("india_ppi_corr_flag",
                F.when(F.col("farm_equip_ppi_yoy_pct") > 5, "High PPI Growth")
                 .when(F.col("farm_equip_ppi_yoy_pct") < -2, "PPI Decline")
                 .otherwise("Normal"))
    .drop("prev_india", "prev_us",
          "_transformed_at", "_silver_layer",
          "avg_corn_ppi_yoy_pct", "avg_wheat_ppi_yoy_pct",
          "avg_farm_equip_ppi_detail")
    .orderBy("year")
)

# ── Add high_farm_income_flag using median ──────────────────────────────────
# Approximate median via percentile_approx, then flag rows above it
median_income = (
    price_vs_sales
    .filter(F.col("net_farm_income_bn_usd").isNotNull())
    .agg(F.percentile_approx("net_farm_income_bn_usd", 0.5).alias("median"))
    .collect()[0]["median"]
)

price_vs_sales = (
    price_vs_sales
    .withColumn("high_farm_income_flag",
                F.col("net_farm_income_bn_usd") >= F.lit(median_income))
    .select(
        "year",
        "india_units", "us_units",
        "avg_farm_equip_ppi", "avg_corn_ppi", "avg_wheat_ppi",
        "net_farm_income_bn_usd",
        "farm_equip_ppi_yoy_pct", "farm_income_yoy_pct",
        "india_units_yoy_pct",    "us_units_yoy_pct",
        "india_ppi_corr_flag",    "high_farm_income_flag"
    )
)

write_gold(price_vs_sales, "price_vs_sales",
           "Demand drivers: equipment PPI, crop prices, farm income vs. India & US units")

spark.table("gold.price_vs_sales") \
    .select("year", "india_units", "us_units",
            "avg_farm_equip_ppi", "net_farm_income_bn_usd",
            "farm_equip_ppi_yoy_pct", "india_units_yoy_pct") \
    .orderBy("year") \
    .show(26, truncate=False)

# ── Pearson correlations (summary print only) ────────────────────────────────
print("\nPearson correlations (printed for reference, not stored):")
corr_df = spark.table("gold.price_vs_sales").filter(
    F.col("india_units").isNotNull() &
    F.col("avg_farm_equip_ppi").isNotNull()
)

pairs = [
    ("india_units", "avg_farm_equip_ppi",     "India units vs Equipment PPI"),
    ("india_units", "net_farm_income_bn_usd", "India units vs US Farm Income"),
    ("us_units",    "avg_farm_equip_ppi",     "US units    vs Equipment PPI"),
    ("us_units",    "net_farm_income_bn_usd", "US units    vs US Farm Income"),
    ("india_units", "avg_corn_ppi",           "India units vs Corn PPI"),
    ("india_units", "avg_wheat_ppi",          "India units vs Wheat PPI"),
]

for col_a, col_b, label in pairs:
    try:
        r = corr_df.stat.corr(col_a, col_b)
        print(f"  {label:<45}  r = {r:+.4f}")
    except Exception as e:
        print(f"  {label:<45}  ERROR: {e}")

## Cell 8 — `gold.top_exporters`

**Source:** `silver.comtrade_hs8701_exports`  
**Grain:** One row per (year, reporter country) for Export flows only.

Two granularities are combined in a single table using a `summary_type` column:
- `annual` — metrics per exporter per year
- `cumulative` — total over the full dataset period (2015–2022)

| Column | Description |
|---|---|
| `summary_type` | `annual` or `cumulative` |
| `year` | Calendar year (`null` for cumulative rows) |
| `reporter_name` | Exporting country |
| `reporter_iso2` | ISO-2 code |
| `total_fob_value_usd` | Total FOB export value |
| `total_fob_value_bn_usd` | Same in $bn (rounded to 2 dp) |
| `total_quantity_units` | Total units exported |
| `avg_usd_per_unit` | Average unit value (price tier proxy) |
| `export_rank` | Rank by FOB value within the year (or overall) |
| `value_share_pct` | Share of world export FOB value |
| `is_top10` | True if in the top-10 exporters overall |

In [ ]:
comtrade = (
    spark.table(f"{SILVER_SCHEMA}.comtrade_hs8701_exports")
    .filter(F.col("flow_code") == "X")   # exports only
    .filter(F.col("reporter_name").isNotNull())
    # Exclude 'World' / EU aggregates from country-level ranking
    .filter(~F.col("reporter_iso2").isin("WLD", "EU"))
)

# ── Annual aggregation ────────────────────────────────────────────────────────
annual_exports = (
    comtrade
    .groupBy("year", "reporter_name", "reporter_iso2")
    .agg(
        F.sum("fob_value_usd").alias("total_fob_value_usd"),
        F.sum("quantity_units").alias("total_quantity_units"),
        F.round(F.avg("usd_per_unit"), 0).alias("avg_usd_per_unit")
    )
)

# World total per year (for share computation)
world_annual = (
    annual_exports
    .groupBy("year")
    .agg(F.sum("total_fob_value_usd").alias("world_fob_usd"))
)

w_annual_rank = Window.partitionBy("year").orderBy(F.desc("total_fob_value_usd"))

annual_ranked = (
    annual_exports
    .join(world_annual, "year", "left")
    .withColumn("export_rank",
                F.rank().over(w_annual_rank))
    .withColumn("value_share_pct",
                F.round(F.col("total_fob_value_usd") / F.col("world_fob_usd") * 100, 2))
    .withColumn("total_fob_value_bn_usd",
                F.round(F.col("total_fob_value_usd") / 1e9, 3))
    .withColumn("summary_type", F.lit("annual"))
    .drop("world_fob_usd")
)

# ── Cumulative aggregation (all years combined) ───────────────────────────────
cumulative_exports = (
    comtrade
    .groupBy("reporter_name", "reporter_iso2")
    .agg(
        F.sum("fob_value_usd").alias("total_fob_value_usd"),
        F.sum("quantity_units").alias("total_quantity_units"),
        F.round(F.avg("usd_per_unit"), 0).alias("avg_usd_per_unit")
    )
)

world_total_fob = cumulative_exports.agg(F.sum("total_fob_value_usd")).collect()[0][0]

w_cum_rank = Window.orderBy(F.desc("total_fob_value_usd"))

cumulative_ranked = (
    cumulative_exports
    .withColumn("year",        F.lit(None).cast(IntegerType()))
    .withColumn("export_rank",
                F.rank().over(w_cum_rank))
    .withColumn("value_share_pct",
                F.round(F.col("total_fob_value_usd") / F.lit(world_total_fob) * 100, 2))
    .withColumn("total_fob_value_bn_usd",
                F.round(F.col("total_fob_value_usd") / 1e9, 3))
    .withColumn("summary_type", F.lit("cumulative"))
)

# ── Identify top-10 by cumulative FOB ────────────────────────────────────────
top10_set = (
    cumulative_ranked
    .filter(F.col("export_rank") <= 10)
    .select("reporter_iso2")
    .withColumn("is_top10", F.lit(True))
)

# ── Union annual + cumulative, join top10 flag ────────────────────────────────
all_cols = [
    "summary_type", "year", "reporter_name", "reporter_iso2",
    "total_fob_value_usd", "total_fob_value_bn_usd",
    "total_quantity_units", "avg_usd_per_unit",
    "export_rank", "value_share_pct"
]

top_exporters = (
    annual_ranked.select(all_cols)
    .union(cumulative_ranked.select(all_cols))
    .join(top10_set, "reporter_iso2", "left")
    .withColumn("is_top10", F.coalesce(F.col("is_top10"), F.lit(False)))
    .orderBy(
        F.col("summary_type").desc(),   # 'cumulative' before 'annual'
        F.col("year").asc_nulls_first(),
        "export_rank"
    )
)

write_gold(top_exporters, "top_exporters",
           "UN Comtrade HS8701 — top exporters annual + cumulative with rank and share")

print("\nTop-10 exporters (cumulative 2015–2022):")
spark.table("gold.top_exporters") \
    .filter((F.col("summary_type") == "cumulative") & F.col("is_top10")) \
    .orderBy("export_rank") \
    .select("export_rank", "reporter_name", "reporter_iso2",
            "total_fob_value_bn_usd", "total_quantity_units",
            "avg_usd_per_unit", "value_share_pct") \
    .show(10, truncate=False)

print("\nTop-5 exporters by year:")
spark.table("gold.top_exporters") \
    .filter((F.col("summary_type") == "annual") & (F.col("export_rank") <= 5)) \
    .orderBy("year", "export_rank") \
    .select("year", "export_rank", "reporter_name",
            "total_fob_value_bn_usd", "value_share_pct") \
    .show(40, truncate=False)

## Cell 9 — `gold.india_brand_share`

**Source:** `silver.india_tractor_brands`  
**Grain:** One row per brand — static snapshot (2023 market share data).

Enriched with:
- Cumulative share % (for waterfall / Pareto charts)
- HP midpoint (useful for scatter plots: share vs. HP focus)
- `segment_label` = short segment description for chart labels
- `brand_tier` = `Tier-1` (>15% share) / `Tier-2` (5–15%) / `Tier-3` (<5%)

| Column | Description |
|---|---|
| `market_share_rank` | 1 = largest |
| `brand` | Brand name |
| `parent_company` | Parent company |
| `country_of_origin` | Country where brand originates |
| `is_indian_brand` | True if Indian origin |
| `market_share_2023_pct` | Raw market share % |
| `market_share_2023_pct_norm` | Normalised share (sums to 100%) |
| `cumulative_share_pct` | Cumulative share for Pareto chart |
| `brand_tier` | Tier-1 / Tier-2 / Tier-3 |
| `hp_min` / `hp_max` / `hp_midpoint` | HP range and midpoint |
| `segment_label` | e.g. `20–75 HP` |
| `key_models` | Key model names |

In [ ]:
brands_silver = spark.table(f"{SILVER_SCHEMA}.india_tractor_brands")

w_rank = Window.orderBy(F.desc("market_share_2023_pct_norm"))

india_brand_share = (
    brands_silver
    # Cumulative share for Pareto analysis
    .withColumn("cumulative_share_pct",
                F.round(
                    F.sum("market_share_2023_pct_norm")
                    .over(Window.orderBy(F.desc("market_share_2023_pct_norm"))
                                .rowsBetween(Window.unboundedPreceding, 0)),
                    2
                ))
    # HP midpoint
    .withColumn("hp_midpoint",
                F.when(
                    F.col("hp_min").isNotNull() & F.col("hp_max").isNotNull(),
                    F.round((F.col("hp_min") + F.col("hp_max")) / 2, 0)
                ).otherwise(F.lit(None).cast(DoubleType())))
    # Segment label
    .withColumn("segment_label",
                F.when(
                    F.col("hp_min").isNotNull() & F.col("hp_max").isNotNull(),
                    F.concat(
                        F.col("hp_min").cast(StringType()),
                        F.lit("–"),
                        F.col("hp_max").cast(StringType()),
                        F.lit(" HP")
                    )
                ).otherwise(F.col("hp_range_focus")))
    # Brand tier
    .withColumn("brand_tier",
                F.when(F.col("market_share_2023_pct_norm") >= 15, "Tier-1")
                 .when(F.col("market_share_2023_pct_norm") >= 5,  "Tier-2")
                 .otherwise("Tier-3"))
    .drop("_transformed_at", "_silver_layer", "notes")
    .select(
        "market_share_rank", "brand", "parent_company",
        "country_of_origin", "is_indian_brand",
        "market_share_2023_pct", "market_share_2023_pct_norm",
        "cumulative_share_pct", "brand_tier",
        "hp_min", "hp_max", "hp_midpoint",
        "hp_range_focus", "segment_label",
        "key_models"
    )
    .orderBy("market_share_rank")
)

write_gold(india_brand_share, "india_brand_share",
           "India brand market share 2023 — ranked, Pareto share, HP midpoint, tiers")

spark.table("gold.india_brand_share") \
    .select("market_share_rank", "brand", "parent_company",
            "market_share_2023_pct_norm", "cumulative_share_pct",
            "brand_tier", "segment_label", "is_indian_brand") \
    .show(20, truncate=False)

# Indian vs foreign brand share split
print("\nIndian vs foreign brand share split:")
spark.table("gold.india_brand_share") \
    .groupBy("is_indian_brand") \
    .agg(
        F.count("*").alias("num_brands"),
        F.round(F.sum("market_share_2023_pct_norm"), 2).alias("total_share_pct")
    ) \
    .show(truncate=False)

## Cell 10 — Data Quality Checks

Automated assertions on all Gold tables before declaring the layer complete.

Checks:
1. Row counts ≥ expected minimums
2. No nulls in key dimension columns
3. Share percentages in valid range [0, 100]
4. Rank columns start at 1 and have no gaps for the top-N
5. `gold.india_seasonality` — monthly units positive for every fiscal year
6. `gold.market_summary` — World totals are the sum of individual countries per year
7. `gold.top_exporters` — cumulative FOB ≈ sum of annual FOB

In [ ]:
from pyspark.sql.utils import AnalysisException

PASS = "PASS "
FAIL = "FAIL "
WARN = "WARN "

results = []

def check(name: str, passed: bool, warn_only: bool = False, detail: str = ""):
    status = PASS if passed else (WARN if warn_only else FAIL)
    results.append((status, name, detail))

# ── 1. Row count minimums ────────────────────────────────────────────────────
min_rows = {
    "gold.market_summary":    50,
    "gold.india_seasonality": 200,
    "gold.hp_segment_trend":  40,
    "gold.price_vs_sales":    20,
    "gold.top_exporters":     10,
    "gold.india_brand_share":  5,
}

for tbl, min_r in min_rows.items():
    try:
        n = spark.table(tbl).count()
        check(f"row_count: {tbl}", n >= min_r, detail=f"{n} rows (min {min_r})")
    except AnalysisException as e:
        check(f"row_count: {tbl}", False, detail=str(e))

# ── 2. No nulls in key dimensions ────────────────────────────────────────────
null_checks = {
    "gold.market_summary":    ["year", "country", "units_sold"],
    "gold.india_seasonality": ["fiscal_year", "fiscal_month", "estimated_monthly_units"],
    "gold.hp_segment_trend":  ["year", "hp_category", "units_sold"],
    "gold.price_vs_sales":    ["year"],
    "gold.top_exporters":     ["reporter_name", "export_rank", "total_fob_value_usd"],
    "gold.india_brand_share": ["brand", "market_share_rank"],
}

for tbl, cols in null_checks.items():
    try:
        df = spark.table(tbl)
        for c in cols:
            null_n = df.filter(F.col(c).isNull()).count()
            check(f"no_nulls: {tbl}.{c}", null_n == 0, detail=f"{null_n} nulls")
    except AnalysisException as e:
        check(f"no_nulls: {tbl}", False, detail=str(e))

# ── 3. Share percentages in [0, 100] ─────────────────────────────────────────
try:
    bad = spark.table("gold.market_summary") \
        .filter(
            (F.col("country_share_pct") < 0) | (F.col("country_share_pct") > 100)
        ).count()
    check("share_range: gold.market_summary.country_share_pct", bad == 0,
          detail=f"{bad} rows out of [0,100]")
except AnalysisException as e:
    check("share_range: gold.market_summary", False, detail=str(e))

try:
    bad = spark.table("gold.india_brand_share") \
        .filter(
            (F.col("market_share_2023_pct_norm") < 0) |
            (F.col("market_share_2023_pct_norm") > 100)
        ).count()
    check("share_range: gold.india_brand_share", bad == 0,
          detail=f"{bad} rows out of [0,100]")
except AnalysisException as e:
    check("share_range: gold.india_brand_share", False, detail=str(e))

# ── 4. India seasonality — all monthly estimates > 0 ─────────────────────────
try:
    neg = spark.table("gold.india_seasonality") \
        .filter(F.col("estimated_monthly_units") <= 0).count()
    check("positive_monthly_units: gold.india_seasonality", neg == 0,
          detail=f"{neg} rows with units ≤ 0")
except AnalysisException as e:
    check("positive_monthly_units: gold.india_seasonality", False, detail=str(e))

# ── 5. market_summary — World totals ≈ sum of countries ──────────────────────
try:
    ms = spark.table("gold.market_summary")
    country_sum = (
        ms.filter(F.col("country") != "World")
        .groupBy("year")
        .agg(F.sum("units_sold").alias("country_sum"))
    )
    world_total = (
        ms.filter(F.col("country") == "World")
        .select("year", F.col("units_sold").alias("world_total"))
    )
    mismatch = (
        country_sum.join(world_total, "year")
        .filter(F.abs(F.col("country_sum") - F.col("world_total")) > 100)
        .count()
    )
    check("world_equals_country_sum: gold.market_summary", mismatch == 0,
          warn_only=True, detail=f"{mismatch} year(s) with >100 unit discrepancy")
except AnalysisException as e:
    check("world_equals_country_sum: gold.market_summary", False, detail=str(e))

# ── 6. top_exporters — cumulative FOB ≈ sum of annual FOB ────────────────────
try:
    te = spark.table("gold.top_exporters")
    annual_sum = (
        te.filter(F.col("summary_type") == "annual")
        .agg(F.sum("total_fob_value_usd").alias("annual_sum"))
        .collect()[0]["annual_sum"]
    )
    cumulative_sum = (
        te.filter(F.col("summary_type") == "cumulative")
        .agg(F.sum("total_fob_value_usd").alias("cumulative_sum"))
        .collect()[0]["cumulative_sum"]
    )
    pct_diff = abs(annual_sum - cumulative_sum) / cumulative_sum * 100 if cumulative_sum else 100
    check("cumulative_fob_matches_annual_sum: gold.top_exporters",
          pct_diff < 1.0, warn_only=True,
          detail=f"annual_sum={annual_sum/1e9:.2f}bn, cumulative_sum={cumulative_sum/1e9:.2f}bn, diff={pct_diff:.2f}%")
except AnalysisException as e:
    check("cumulative_fob_matches_annual_sum: gold.top_exporters", False, detail=str(e))

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'STATUS':<6}  {'CHECK':<65}  DETAIL")
print("-" * 110)
fail_count = 0
for status, name, detail in results:
    print(f"{status}  {name:<65}  {detail}")
    if status == FAIL:
        fail_count += 1

print("-" * 110)
print(f"\nTotal checks: {len(results)}  |  Failures: {fail_count}")
if fail_count > 0:
    raise Exception(f"{fail_count} DQ check(s) failed — review output above.")
else:
    print("All Gold DQ checks passed.")

## Cell 11 — Gold Layer Summary

Print a final inventory of all Gold tables with row counts and column counts,
and generate the key narrative statistics used in the web report headline cards.

In [ ]:
GOLD_TABLES = [
    ("gold.market_summary",    "Global market: country-year volumes, share %, YoY, CAGR"),
    ("gold.india_seasonality", "India monthly estimated units with Kharif/Rabi labels"),
    ("gold.hp_segment_trend",  "US HP segment trend — long format with YoY per segment"),
    ("gold.price_vs_sales",    "Demand drivers: PPI, farm income vs. India & US units"),
    ("gold.top_exporters",     "UN Comtrade top exporters — annual + cumulative, ranked"),
    ("gold.india_brand_share", "India brand market share 2023 — Pareto, tiers, HP range"),
]

print(f"\n{'TABLE':<35}  {'ROWS':>8}  {'COLS':>5}  DESCRIPTION")
print("-" * 100)

for table, desc in GOLD_TABLES:
    try:
        df   = spark.table(table)
        rows = df.count()
        cols = len(df.columns)
        print(f"{table:<35}  {rows:>8,}  {cols:>5}  {desc}")
    except Exception as e:
        print(f"{table:<35}  {'ERROR':>8}         {e}")

print("-" * 100)

## Cell 12 — Headline KPI Snapshot

Compute the four headline numbers used in the landing-page KPI cards of the Next.js report.

In [ ]:
# ── 1. World total (latest year) ─────────────────────────────────────────────
ms = spark.table("gold.market_summary")
latest_year = ms.agg(F.max("year")).collect()[0][0]

world_latest = (
    ms.filter((F.col("country") == "World") & (F.col("year") == latest_year))
    .select("units_sold", "yoy_growth_pct", "cagr_5yr_pct")
    .collect()[0]
)

# ── 2. India rank among all countries (latest year) ──────────────────────────
india_rank = (
    ms.filter((F.col("year") == latest_year) & (F.col("country") != "World"))
    .orderBy(F.desc("units_sold"))
    .select("country", "units_sold", "country_share_pct")
    .limit(5)
    .collect()
)

# ── 3. India peak seasonality month ──────────────────────────────────────────
peak_month = (
    spark.table("gold.india_seasonality")
    .groupBy("month_name", "season_group")
    .agg(F.avg("avg_share_pct").alias("avg_share"))
    .orderBy(F.desc("avg_share"))
    .first()
)

# ── 4. Top exporter and its cumulative share ──────────────────────────────────
top_exporter = (
    spark.table("gold.top_exporters")
    .filter((F.col("summary_type") == "cumulative") & (F.col("export_rank") == 1))
    .select("reporter_name", "total_fob_value_bn_usd", "value_share_pct")
    .first()
)

# ── Print headline card values ────────────────────────────────────────────────
print("=" * 60)
print("  HEADLINE KPI CARDS  (for Next.js landing page)")
print("=" * 60)
print(f"  Global Market ({latest_year})")
print(f"    Total units sold : {world_latest['units_sold']:,}")
print(f"    YoY growth       : {world_latest['yoy_growth_pct']}%")
print(f"    5-year CAGR      : {world_latest['cagr_5yr_pct']}%")
print()
print(f"  Top Countries ({latest_year}):")
for i, r in enumerate(india_rank, 1):
    print(f"    #{i} {r['country']:<20} {r['units_sold']:>8,} units  ({r['country_share_pct']}%)")
print()
print(f"  India Seasonality Peak Month")
print(f"    Month      : {peak_month['month_name']}")
print(f"    Avg share  : {peak_month['avg_share']:.1f}%")
print(f"    Season     : {peak_month['season_group']}")
print()
if top_exporter:
    print(f"  Top Global Exporter (2015–2022 cumulative)")
    print(f"    Country       : {top_exporter['reporter_name']}")
    print(f"    FOB value     : ${top_exporter['total_fob_value_bn_usd']:.2f}bn")
    print(f"    World share   : {top_exporter['value_share_pct']}%")
print("=" * 60)

## Cell 13 — Next Steps

Gold layer is complete. All six KPI tables are ready in the `gold` schema.

### Dashboard consumption

Connect **Looker Studio** to Databricks SQL and build the following pages:

| Dashboard Page | Gold Table(s) | Chart Types |
|---|---|---|
| Overview — Global Market | `gold.market_summary` (World row) | KPI scorecard, line chart |
| India Deep-Dive | `gold.india_seasonality`, `gold.india_brand_share` | Bar + line, pie/donut |
| US Market | `gold.hp_segment_trend` | Stacked area, bar |
| Global Trade Map | `gold.top_exporters` | Choropleth, horizontal bar |
| Demand Drivers | `gold.price_vs_sales` | Dual-axis line, scatter |
| Seasonality Heatmap | `gold.india_seasonality` | Calendar heatmap |

### Next.js web report

Export Gold tables as JSON (via Databricks REST SQL API or manual CSV export) and place in:
```
tractor-insights/public/data/
  market_summary.json
  india_seasonality.json
  hp_segment_trend.json
  price_vs_sales.json
  top_exporters.json
  india_brand_share.json
```

Then build the Next.js pages using Recharts for client-side rendering.

```python
# Quick export snippet (run in this notebook or a separate export cell)
for table_name in [
    "market_summary", "india_seasonality", "hp_segment_trend",
    "price_vs_sales", "top_exporters", "india_brand_share"
]:
    (
        spark.table(f"gold.{table_name}")
        .drop("_aggregated_at", "_gold_layer")
        .coalesce(1)
        .write
        .mode("overwrite")
        .json(f"/FileStore/tractor/gold_export/{table_name}")
    )
    print(f"Exported gold.{table_name}")
```